# 📖 Notebook 2: Content Deduplication & Ranking

When a major story breaks, dozens of news outlets publish articles about it within minutes. A good news aggregator must **detect these duplicates** and pick the best version to show. Then it must **rank** all articles so the most important, freshest stories appear first.

## Learning Objectives

By the end of this notebook, you'll understand:
- How exact deduplication works using content hashes (SHA-256)
- How near-duplicate detection works using shingling and Jaccard similarity
- How to group duplicates and pick a canonical article
- How to rank articles by freshness, popularity, and a combined score

## 🛠️ Setup

```bash
cd 06-system-designs/news-aggregator
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import hashlib
import json
import time
import math
from datetime import datetime, timezone

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "newsagg_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

conn = get_db(); conn.close(); print("✅ PostgreSQL")
r = get_redis(); r.ping(); print("✅ Redis")

## 🔍 The Duplicate Problem

Let's look at our seed data. Two pairs of articles report the same story:

1. **GPT-5 release** — covered by TechCrunch and Ars Technica
2. **Lakers championship** — covered by ESPN and BBC

A user shouldn't see both versions. We need to detect these duplicates.

In [ ]:
conn = get_db()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Find articles with the same content_hash — these are exact duplicates
cursor.execute("""
    SELECT content_hash, COUNT(*) as count,
           array_agg(title) as titles,
           array_agg(id) as ids
    FROM articles
    GROUP BY content_hash
    HAVING COUNT(*) > 1
    ORDER BY count DESC
""")

dupes = cursor.fetchall()
print(f"🔍 Found {len(dupes)} duplicate clusters by content hash:")
print("=" * 80)
for group in dupes:
    print(f"\n  Hash: {group['content_hash'][:16]}...  ({group['count']} articles)")
    for title, aid in zip(group['titles'], group['ids']):
        print(f"    [{aid:>2}] {title[:65]}")

conn.close()

## 🔑 Strategy 1: Exact Dedup With a Content Hash

The simplest dedup technique: compute a **SHA-256 hash** of the article's *normalised* text (lowercase, punctuation stripped, whitespace collapsed). Two articles get the same hash only when their normalised text is **byte-for-byte identical**.

```
Article A: "OpenAI Releases GPT-5!"  ─┐
                 normalise ──►  "openai releases gpt5"
                       SHA-256 ──►  "8f41...e1"

Article B: "OPENAI RELEASES GPT-5."  ─┘  same normalised text → same hash ✅

Article C: "GPT-5 Launches..."       ──►  different words → different hash ❌
```

> ⚠️ **This is "exact dedup after normalisation", not "near-duplicate" detection.** Any change in wording — even a synonym or a reordered sentence — produces a different hash. We'll handle rewrites in Strategy 2.

**What it's good for**
- Detecting the *same article republished* by a syndication partner
- Catching when a crawler ingests the same feed twice

**What it misses**
- Five outlets rewriting the same story in their own words (the real news-aggregator problem)


In [ ]:
import re

def compute_content_hash(title: str, summary: str) -> str:
    """
    Create a SHA-256 hash from normalised title + summary.
    Normalisation: lowercase, remove punctuation, collapse whitespace.
    """
    text = f"{title} {summary}".lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)  # keep only letters, numbers, spaces
    text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
    return hashlib.sha256(text.encode()).hexdigest()

# Demo: identical articles get the same hash
hash_a = compute_content_hash(
    "Breaking: Major Discovery on Mars",
    "Scientists found water on Mars today."
)
hash_b = compute_content_hash(
    "BREAKING: Major Discovery on Mars!",
    "Scientists found water on Mars today!"
)
hash_c = compute_content_hash(
    "Water Found on Mars in Historic Discovery",
    "Researchers announce the detection of water on the red planet."
)

print("🔑 Content Hash Comparison")
print("=" * 60)
print(f"  Article A: {hash_a[:24]}...")
print(f"  Article B: {hash_b[:24]}...")
print(f"  Article C: {hash_c[:24]}...")
print()
print(f"  A == B? {hash_a == hash_b}  ← same story, minor formatting differences")
print(f"  A == C? {hash_a == hash_c}  ← same story, completely rewritten")
print()
print("💡 Exact hashing catches A≈B but misses A≈C. We need fuzzy matching too.")

## 🧩 Strategy 2: Fuzzy Dedup With Bag-of-Words Jaccard

When five outlets cover the same story, they use **different words** — but they still mention the same *key terms*: names, places, numbers, the event itself.

The beginner-friendly fuzzy dedup trick:
1. Tokenise each article into a **set of meaningful words** (drop stop-words like "the", "is", "of")
2. Compare two sets with **Jaccard similarity**:

```
Jaccard(A, B) = |A ∩ B| / |A ∪ B|
              = (words in both)   /  (words in either)
```

Jaccard is `0` when the sets share nothing, `1` when they're identical. A score above a threshold (often ~0.15–0.30 for short news summaries) usually means "same story".

### Why this works when hashing doesn't
The two GPT-5 articles share key terms — *openai, gpt, reasoning, major* — even though their sentences are rewritten. A set of words ignores word order, so paraphrasing barely hurts the score.


In [ ]:
# Simple stopword list + a tiny "news boilerplate" list.
# Production systems compute these from corpus frequency. For a teaching
# notebook, a small hand-picked list is plenty.
STOPWORDS = set("""
a an the is are was were be been being am of for to in on at by with from as
and or but if this that these those it its their there here has have had do
does did will would can could should may might must not no yes so than then
when where which who whom whose why how what about into over under again very
""".split())
BOILERPLATE = set("breaking live update updates report story news today article url comments points https http www com org net via read more".split())

URL_RE = re.compile(r"https?://\S+")

def tokenise(text: str) -> set:
    """Return a set of meaningful lowercase words from a piece of text."""
    text = URL_RE.sub(" ", text)                           # strip URLs first
    text = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    return {
        w for w in text.split()
        if len(w) > 2 and w not in STOPWORDS and w not in BOILERPLATE
    }


def jaccard_similarity(set_a: set, set_b: set) -> float:
    """Jaccard = |intersection| / |union|. Ranges from 0 to 1."""
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


# --- Demo on our seed articles ---
article_1 = (
    "OpenAI Releases GPT-5 With Reasoning Breakthrough. "
    "OpenAI announced GPT-5 today, featuring a major leap in multi-step reasoning "
    "capabilities. The model can solve complex math and science problems with 95% accuracy."
)
article_2 = (
    "GPT-5 Launches With Major Reasoning Improvements. "
    "OpenAI has released GPT-5. The new model demonstrates significant improvements "
    "in chain-of-thought reasoning, scoring 95% on graduate-level math benchmarks."
)
article_unrelated = (
    "Lakers Win NBA Championship in Game 7 Thriller. "
    "The Los Angeles Lakers defeated the Boston Celtics 108-105 in a dramatic "
    "Game 7 to claim their 18th NBA championship."
)

w1, w2, w3 = tokenise(article_1), tokenise(article_2), tokenise(article_unrelated)

sim_dupe      = jaccard_similarity(w1, w2)
sim_unrelated = jaccard_similarity(w1, w3)

print("🧩 Bag-of-Words Jaccard Similarity")
print("=" * 55)
print(f"  Article 1 tokens: {len(w1)}")
print(f"  Article 2 tokens: {len(w2)}")
print(f"  Shared tokens:    {len(w1 & w2)}  →  {sorted(w1 & w2)}")
print()
print(f"  Similarity (GPT-5 v GPT-5):   {sim_dupe:.3f}  ← same story, rewritten")
print(f"  Similarity (GPT-5 v Lakers):  {sim_unrelated:.3f}  ← unrelated")


In [ ]:
# How do we pick a threshold? By looking at scores for known pairs.
pairs = [
    ("GPT-5 TechCrunch",   "GPT-5 Ars Technica",   article_1, article_2, "same story"),
    ("GPT-5",              "Lakers",               article_1, article_unrelated, "unrelated"),
    ("GPT-5",              "GPT-5 (identical)",    article_1, article_1, "exact copy"),
    ("Lakers ESPN",        "Lakers BBC",
     "Lakers Win NBA Championship in Game 7 Thriller. LeBron James scored 42 points as the Lakers beat the Celtics.",
     "Los Angeles Lakers Clinch NBA Title With Game 7 Victory. LeBron James led with 42 points over the Celtics.",
     "same story"),
]

print(f"  {'A':<22} {'B':<22} {'Score':>6}  verdict")
print("-" * 70)
for a_name, b_name, a_text, b_text, label in pairs:
    s = jaccard_similarity(tokenise(a_text), tokenise(b_text))
    print(f"  {a_name:<22} {b_name:<22} {s:>6.3f}  ({label})")

print()
print("💡 Threshold ~0.15 separates 'same story' (>= 0.17) from 'unrelated' (0.00)")
print("   on this toy corpus. Real systems tune the threshold empirically.")


## 🏗️ Building Duplicate Groups

When we detect duplicates, we group them together and pick a **canonical** article — the one we'll show in the feed. The others are hidden but their existence boosts the story's importance ("reported by 5 sources").

How to pick the canonical article:
- **Most detailed** — highest word count
- **First published** — the original reporter
- **Most trusted source** — from a tier-1 outlet

Let's implement this against our database.

In [ ]:
MIN_SHARED_TOKENS = 3   # guard against false positives on tiny token sets

def find_near_duplicates(threshold: float = 0.15, min_shared: int = MIN_SHARED_TOKENS):
    """
    Detect near-duplicate articles using bag-of-words Jaccard similarity.

    Returns (groups, info) where:
      - groups : list of lists of article_ids forming a duplicate cluster
      - info   : dict {article_id: {"tokens": set, "article": row}}

    Guard against over-merging: a new article only joins an existing cluster
    if its similarity to the cluster's *representative* is also >= threshold.
    Pure union-find alone can transitively link A~B and B~C even when A !~ C.
    """
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute("""
        SELECT id, title, summary, word_count, published_at
        FROM articles
        ORDER BY published_at DESC
    """)
    articles = cursor.fetchall()
    conn.close()

    # Build token sets
    info = {}
    for a in articles:
        text = f"{a['title']} {a['summary'] or ''}"
        info[a['id']] = {"tokens": tokenise(text), "article": a}

    # Compare all pairs (O(n^2) — fine for a few hundred articles; see MinHash below)
    ids = list(info.keys())
    pairs = []
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            ta, tb = info[ids[i]]["tokens"], info[ids[j]]["tokens"]
            # Skip ultra-short docs: Jaccard is unreliable on sets of 2–3 tokens
            if len(ta) < 5 or len(tb) < 5:
                continue
            shared = len(ta & tb)
            if shared < min_shared:
                continue
            s = jaccard_similarity(ta, tb)
            if s >= threshold:
                pairs.append((ids[i], ids[j], s))

    # Greedy clustering with a "match the representative" guard
    clusters = []            # each cluster = [rep_id, member_ids...]
    pairs.sort(key=lambda p: -p[2])   # strongest pairs first
    assigned = {}

    for a, b, s in pairs:
        ca, cb = assigned.get(a), assigned.get(b)
        if ca is None and cb is None:
            clusters.append([a, b])
            assigned[a] = assigned[b] = len(clusters) - 1
        elif ca is not None and cb is None:
            rep = clusters[ca][0]
            tr, tb = info[rep]["tokens"], info[b]["tokens"]
            if len(tr & tb) >= min_shared and jaccard_similarity(tr, tb) >= threshold:
                clusters[ca].append(b); assigned[b] = ca
        elif cb is not None and ca is None:
            rep = clusters[cb][0]
            tr, ta = info[rep]["tokens"], info[a]["tokens"]
            if len(tr & ta) >= min_shared and jaccard_similarity(tr, ta) >= threshold:
                clusters[cb].append(a); assigned[a] = cb
        # both already assigned to different clusters: leave alone, no merge

    groups = [c for c in clusters if len(c) > 1]
    return groups, info


dup_groups, info = find_near_duplicates(threshold=0.15)

print(f"🔍 Found {len(dup_groups)} duplicate groups:")
print("=" * 70)
for i, group in enumerate(dup_groups):
    print(f"\n  Group {i+1}:")
    for aid in group:
        a = info[aid]["article"]
        print(f"    [{aid:>3}] {a['title'][:55]:<55} ({a['word_count']} words)")

print("\n💡 The seeded GPT-5 and Lakers clusters should appear above.")


In [ ]:
def pick_canonical(group_ids: list, info: dict) -> int:
    """
    Pick the canonical article to display from a duplicate cluster.
    Strategy: longest summary (most detail). Tie-break: earliest published.
    """
    best = None
    for aid in group_ids:
        a = info[aid]["article"]
        if best is None:
            best = a
        elif a["word_count"] > best["word_count"]:
            best = a
        elif (a["word_count"] == best["word_count"]
              and a["published_at"] and best["published_at"]
              and a["published_at"] < best["published_at"]):
            best = a
    return best["id"]

print("🏆 Canonical article per duplicate cluster:")
print("=" * 70)
for i, group in enumerate(dup_groups):
    canonical_id = pick_canonical(group, info)
    print(f"\n  Group {i+1} — canonical: article #{canonical_id}")
    for aid in group:
        a = info[aid]["article"]
        marker = "  ★ SHOW" if aid == canonical_id else "    hide"
        print(f"  {marker} [{aid:>3}] {a['title'][:50]} ({a['word_count']} words)")

print("\n💡 The user sees one article per story, but we record")
print("   'reported by N sources' so popular stories rank higher.")


### 💾 Persist What We Detected (Closing the Loop)

Up to here the detection has been a dead end: we found clusters in memory and
threw them away. The feed builder below reads `duplicate_groups` /
`duplicate_members`, which so far only contain **hand-written seed rows**.

That gap is worth naming, because it's a common way a pipeline quietly stops
working: the detector improves, and nothing downstream notices. Let's write our
detected clusters to the database and derive `source_count` from them instead of
trusting the seeded value.

Two things to get right:

- **Idempotency.** The dedup job runs on a schedule. Re-running it must not
  create a second group for the same story, so we clear the groups we own before
  rewriting them.
- **`source_count` is derived, not declared.** It should equal the number of
  *distinct feeds* in the cluster — not the number of articles. Two articles
  from the same outlet are one source.

In [ ]:
def persist_duplicate_groups(groups, info):
    """Write detected clusters to duplicate_groups / duplicate_members and
    recompute source_count from the number of DISTINCT feeds per cluster.

    Idempotent: wipes the existing groups first, so re-running the dedup job
    converges instead of accumulating.
    """
    conn = get_db()
    conn.autocommit = False
    cur = conn.cursor()
    try:
        # Start from a clean slate — this job owns these two tables.
        cur.execute("DELETE FROM duplicate_members")
        cur.execute("DELETE FROM duplicate_groups")
        # Every article is its own single source until proven otherwise.
        cur.execute("UPDATE articles SET source_count = 1")

        written = 0
        for group in groups:
            canonical = pick_canonical(group, info)
            cur.execute(
                "INSERT INTO duplicate_groups (canonical_article_id) VALUES (%s) RETURNING id",
                (canonical,),
            )
            group_id = cur.fetchone()[0]

            rep_tokens = info[canonical]["tokens"]
            for aid in group:
                sim = 1.0 if aid == canonical else jaccard_similarity(
                    rep_tokens, info[aid]["tokens"])
                cur.execute(
                    "INSERT INTO duplicate_members (group_id, article_id, similarity_score)"
                    " VALUES (%s, %s, %s)",
                    (group_id, aid, round(sim, 2)),
                )

            # source_count = distinct FEEDS in the cluster, not article count.
            cur.execute(
                """UPDATE articles SET source_count = (
                       SELECT COUNT(DISTINCT a2.feed_id)
                       FROM duplicate_members dm2
                       JOIN articles a2 ON a2.id = dm2.article_id
                       WHERE dm2.group_id = %s)
                   WHERE id = ANY(%s)""",
                (group_id, group),
            )
            written += 1
        conn.commit()
        return written
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


written = persist_duplicate_groups(dup_groups, info)
print(f"💾 Persisted {written} detected duplicate group(s)")

# Run it twice — a scheduled job must be idempotent.
persist_duplicate_groups(dup_groups, info)

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT COUNT(*) AS n FROM duplicate_groups")
n_groups = cur.fetchone()["n"]
cur.execute("""
    SELECT dg.id, dg.canonical_article_id, a.title, a.source_count,
           COUNT(dm.article_id) AS members
    FROM duplicate_groups dg
    JOIN articles a ON a.id = dg.canonical_article_id
    JOIN duplicate_members dm ON dm.group_id = dg.id
    GROUP BY dg.id, dg.canonical_article_id, a.title, a.source_count
    ORDER BY dg.id
""")
rows = cur.fetchall()
conn.close()

assert n_groups == written, f"re-running the job duplicated groups: {n_groups} != {written}"
print(f"✅ Ran twice, still {n_groups} group(s) — the job is idempotent.\n")

print(f"  {'grp':>3}  {'members':>7}  {'sources':>7}  canonical")
print("-" * 74)
for row in rows:
    print(f"  {row['id']:>3}  {row['members']:>7}  {row['source_count']:>7}  "
          f"{row['title'][:44]}")
print()
print("💡 source_count is now DERIVED from the clusters we detected. Before this")
print("   cell it was a hand-written seed value — the ranking was trusting a")
print("   number nothing in the pipeline actually maintained.")
print()
print("⚠️  Look at the clusters above with sources=1. Those are two articles from")
print("   the SAME feed that our 0.15 threshold merged. Sometimes that's correct")
print("   (a live-blog reposted), sometimes it's a false positive that just hid a")
print("   real story from every user. Merging is destructive and silent, so a")
print("   production dedup job keeps the hidden articles queryable and tracks a")
print("   merge-rate metric — a threshold change that suddenly merges 3x more")
print("   stories is an outage, not an improvement.")

## 🔠 Aside: Shingling Captures Word *Order*

Bag-of-words throws away word order. Two very different articles that happen to share the same vocabulary ("the dog bit the man" vs "the man bit the dog") get identical token sets.

**Shingling** (aka **n-grams**) keeps short word sequences:

```
"the cat sat on the mat"  →  3-shingles:
  {"the cat sat", "cat sat on", "sat on the", "on the mat"}
```

Two articles sharing many shingles reliably report the same story *in a similar way* — but shingling is **brittle under paraphrasing**. If outlets rewrite every sentence, shingle sets barely overlap.

For short news summaries, bag-of-words Jaccard is usually the better starting point; shingling shines on longer, less-rewritten text. Real systems often try both and pick the higher score.


In [ ]:
def make_shingles(text: str, k: int = 2) -> set:
    """Break text into k-word overlapping shingles."""
    text = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    words = text.split()
    if len(words) < k:
        return {" ".join(words)}
    return {" ".join(words[i:i+k]) for i in range(len(words) - k + 1)}

# Compare shingle-Jaccard vs word-Jaccard on the GPT-5 pair
shingle_sim = jaccard_similarity(make_shingles(article_1), make_shingles(article_2))
word_sim    = jaccard_similarity(tokenise(article_1), tokenise(article_2))

print(f"  Word-set Jaccard    (captures vocabulary):     {word_sim:.3f}")
print(f"  2-shingle Jaccard   (captures short phrases):  {shingle_sim:.3f}")
print()
print("💡 On heavily rewritten news, words overlap but phrases often don't.")


## ⚡ Scaling Up: MinHash + LSH (Conceptual)

Our `find_near_duplicates` compares every pair — that's **O(n²)**. With 1 million articles, that's 500 billion comparisons. Impossible.

**MinHash** lets you *estimate* Jaccard similarity from a tiny fixed-size signature (e.g. 128 integers per article). Two signatures agree in a cell with probability ≈ the true Jaccard similarity, so you can compare signatures in **O(k)** time regardless of document length.

**LSH (Locality-Sensitive Hashing)** then *groups candidates* so you only compare articles that land in the same bucket — turning O(n²) into roughly O(n).

```
 Each article ──► MinHash signature (128 ints)
                      │
                      ▼
                  LSH buckets
                      │  (split signature into bands; bucket by band)
                      ▼
   Only compare articles that share a bucket  ≈  O(n) pairs
```

Related: **SimHash** (used by Google for web-page dedup) produces a fixed-size hash where similar documents have small Hamming distance.


In [ ]:
# A tiny, readable MinHash that shows the *idea* — not a production impl.
import random

def minhash_signature(token_set: set, num_hashes: int = 128, seed: int = 42) -> list:
    """
    Compute a length-`num_hashes` signature. Each slot = min hash value
    of the tokens under a different hash function. Signatures of two sets
    agree in each slot with probability approx Jaccard(set_a, set_b).
    """
    rng = random.Random(seed)
    coefs = [(rng.randint(1, 2**31), rng.randint(0, 2**31)) for _ in range(num_hashes)]
    PRIME = 2**31 - 1

    def h(token, a, b):
        # deterministic hash(token) → int, combined with (a, b)
        return (a * (hash(token) & 0xFFFFFFFF) + b) % PRIME

    sig = []
    for a, b in coefs:
        sig.append(min(h(t, a, b) for t in token_set) if token_set else 0)
    return sig


def estimate_jaccard(sig_a: list, sig_b: list) -> float:
    """Fraction of slots where two signatures agree."""
    return sum(1 for x, y in zip(sig_a, sig_b) if x == y) / len(sig_a)


sig_1 = minhash_signature(w1)
sig_2 = minhash_signature(w2)
sig_3 = minhash_signature(w3)

print("MinHash vs true Jaccard")
print("=" * 45)
print(f"  GPT-5 vs GPT-5   true={jaccard_similarity(w1,w2):.3f}  minhash={estimate_jaccard(sig_1,sig_2):.3f}")
print(f"  GPT-5 vs Lakers  true={jaccard_similarity(w1,w3):.3f}  minhash={estimate_jaccard(sig_1,sig_3):.3f}")
print()
print("💡 128 ints gives a fast, reasonably-accurate estimate.")
print("   In production, pair MinHash with LSH for O(n) candidate generation.")


### Now Actually Run LSH — Because "O(n) in theory" Is Not a Result

Everything above about LSH is a claim. Let's build a synthetic corpus big enough
that O(n²) genuinely hurts, run banded LSH over MinHash signatures, and measure
the two numbers that decide whether it's usable:

- **Candidate reduction** — how many pairs do we actually have to compare?
- **Recall** — of the duplicate pairs we *planted*, how many does LSH surface?

Recall matters more than reduction. An LSH configuration that eliminates 99.9%
of comparisons and also loses a third of the duplicates is not a win; it's a
silent regression. Because we generate the corpus ourselves, we know the ground
truth and can measure recall honestly.

In [ ]:
# ── Build a synthetic corpus with KNOWN duplicate pairs ─────────────────
rng = random.Random(7)

VOCAB = [f"term{i}" for i in range(4000)]
N_STORIES = 600          # distinct stories
N_DUPES = 300            # of which this many get a rewritten sibling
TOKENS_PER_DOC = 30
REWRITE_KEEP = 0.6       # a "rewrite" keeps 60% of the original terms

docs = {}                # doc_id -> token set
truth_pairs = set()      # the duplicate pairs we planted

for s in range(N_STORIES):
    base = set(rng.sample(VOCAB, TOKENS_PER_DOC))
    docs[f"s{s}"] = base
    if s < N_DUPES:
        keep = set(rng.sample(sorted(base), int(TOKENS_PER_DOC * REWRITE_KEEP)))
        rewrite = keep | set(rng.sample(VOCAB, TOKENS_PER_DOC - len(keep)))
        docs[f"s{s}b"] = rewrite
        truth_pairs.add(tuple(sorted((f"s{s}", f"s{s}b"))))

doc_ids = list(docs)
n = len(doc_ids)
brute_force_pairs = n * (n - 1) // 2

planted_sims = [jaccard_similarity(docs[a], docs[b]) for a, b in truth_pairs]
mean_planted = sum(planted_sims) / len(planted_sims)
print(f"Corpus: {n:,} docs, {len(truth_pairs)} planted duplicate pairs")
print(f"Planted-pair Jaccard: min={min(planted_sims):.2f} "
      f"mean={mean_planted:.2f} max={max(planted_sims):.2f}")
print(f"Brute force would need {brute_force_pairs:,} comparisons.\n")

# ── MinHash signatures ──────────────────────────────────────────────────
NUM_HASHES = 64
PRIME = (1 << 31) - 1
coefs = [(rng.randrange(1, PRIME), rng.randrange(0, PRIME)) for _ in range(NUM_HASHES)]

def signature(tokens):
    base = [hash(t) & 0x7FFFFFFF for t in tokens]
    return tuple(min((a * h + b) % PRIME for h in base) for a, b in coefs)

t0 = time.perf_counter()
sigs = {d: signature(docs[d]) for d in doc_ids}
print(f"MinHash: {NUM_HASHES} hashes for {n:,} docs in "
      f"{(time.perf_counter() - t0) * 1000:,.0f} ms\n")

# ── LSH banding, swept across every split of the 64 slots ───────────────
# Split the signature into BANDS bands of ROWS rows. Two docs are candidates
# if ANY band matches exactly. P(candidate) = 1 - (1 - s^ROWS)^BANDS — an
# S-curve whose knee sits near s ≈ (1/BANDS)^(1/ROWS).
THRESHOLD = 0.30

def run_lsh(bands, rows):
    buckets = {}
    for d in doc_ids:
        sig = sigs[d]
        for b in range(bands):
            buckets.setdefault((b, sig[b * rows:(b + 1) * rows]), []).append(d)

    candidates = set()
    for members in buckets.values():
        if len(members) < 2:
            continue
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                candidates.add(tuple(sorted((members[i], members[j]))))

    # LSH narrows WHAT you compare; it does not replace the comparison.
    found = {p for p in candidates
             if jaccard_similarity(docs[p[0]], docs[p[1]]) >= THRESHOLD}
    return candidates, found

print(f"LSH sweep over the {NUM_HASHES}-slot signature "
      f"(planted pairs average Jaccard {mean_planted:.2f}):")
print("=" * 84)
print(f"  {'bands':>5} x {'rows':<5} {'knee':>6}  {'candidates':>12} "
      f"{'vs brute':>10}  {'recall':>7}  {'false pos':>10}")
print("-" * 84)

results = {}
for bands, rows in [(8, 8), (16, 4), (32, 2), (64, 1)]:
    candidates, found = run_lsh(bands, rows)
    recall = len(found & truth_pairs) / len(truth_pairs)
    results[(bands, rows)] = (candidates, recall)
    knee = (1 / bands) ** (1 / rows)
    print(f"  {bands:>5} x {rows:<5} {knee:>6.2f}  {len(candidates):>12,} "
          f"{len(candidates) / brute_force_pairs:>9.2%}  {recall:>6.1%}  "
          f"{len(found - truth_pairs):>10,}")

BEST = (32, 2)
best_candidates, best_recall = results[BEST]
assert best_recall > 0.95, f"{BEST} recall dropped to {best_recall:.1%}"

print()
print(f"✅ At {BEST[0]}x{BEST[1]}: {best_recall:.1%} recall while comparing only "
      f"{len(best_candidates):,} of")
print(f"   {brute_force_pairs:,} pairs — a "
      f"{brute_force_pairs / len(best_candidates):,.0f}x reduction.")
print()
print("💡 Read the sweep as a trade-off curve, not a leaderboard:")
print("   • 8x8  — knee at 0.77, far above our duplicates' 0.43. Almost no")
print("     candidates, and it MISSES nearly every real duplicate. This is the")
print("     dangerous configuration: it looks fast and it is silently broken.")
print("   • 64x1 — knee at 0.02. Catches everything and drags in a pile of")
print("     unrelated pairs you then have to verify one by one.")
print("   • Tune the knee to sit just BELOW the similarity of the duplicates you")
print("     care about — which means you have to know that number, which means")
print("     you need labelled pairs. Recall is not optional to measure.")

## 📊 Ranking Articles

After dedup, we need to decide the **order** articles appear in. The three main signals:

| Signal | What It Means | How We Measure It |
|--------|--------------|-------------------|
| **Freshness** | How new is the article? | Exponential time decay |
| **Popularity** | How widely reported / read? | Source count + interaction count |
| **Relevance** | How relevant to this user? | (Covered in Notebook 3) |

### Freshness Score

We use **exponential decay**: an article's freshness drops by half every 6 hours.

```
freshness = e^(-λ × hours_old)

where λ = ln(2) / half_life_hours
```

This means:
- 0 hours old → score 1.0
- 6 hours old → score 0.5
- 12 hours old → score 0.25
- 24 hours old → score 0.06

In [ ]:
HALF_LIFE_HOURS = 6  # freshness halves every 6 hours
DECAY_RATE = math.log(2) / HALF_LIFE_HOURS

def freshness_score(published_at: datetime) -> float:
    """
    Exponential decay based on article age.
    Returns a value between 0 (very old) and 1 (just published).
    """
    if not published_at:
        return 0.0
    now = datetime.now(timezone.utc)
    # Handle timezone-naive datetimes from the database
    if published_at.tzinfo is None:
        published_at = published_at.replace(tzinfo=timezone.utc)
    age_hours = (now - published_at).total_seconds() / 3600
    return math.exp(-DECAY_RATE * max(age_hours, 0))

# Visualise the decay curve
print("📉 Freshness Decay Curve (half-life = 6 hours):")
print("=" * 55)
for hours in [0, 1, 3, 6, 12, 24, 48]:
    score = math.exp(-DECAY_RATE * hours)
    bar = "█" * int(score * 40)
    print(f"  {hours:>3}h old → {score:.3f}  {bar}")

In [ ]:
def popularity_score(source_count: int, interaction_count: int) -> float:
    """
    Score based on how many sources reported the story and user engagement.
    Uses log scale so 100 interactions isn't 100× better than 1.
    """
    # Source count is very valuable — widely reported = important.
    # log(1+n)/log(10) reaches 1.0 at n=9, so CLAMP it: without min(), a story
    # carried by 50 outlets scores 1.7 and silently outweighs the freshness
    # term, which is capped at 1.0. Un-clamped "normalised" scores are one of
    # the easiest ways to break a weighted ranking without noticing.
    source_score = min(1.0, math.log(1 + source_count) / math.log(10))
    interaction_score = min(1.0, math.log(1 + interaction_count) / math.log(100))
    return 0.6 * source_score + 0.4 * interaction_score


def combined_score(published_at, source_count, interaction_count,
                   freshness_weight=0.6, popularity_weight=0.4) -> float:
    """
    Blend freshness and popularity into a single ranking score.
    """
    f = freshness_score(published_at)
    p = popularity_score(source_count, interaction_count)
    return freshness_weight * f + popularity_weight * p


# Rank all articles in the database
conn = get_db()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cursor.execute("""
    SELECT a.id, a.title, a.published_at, a.source_count, a.content_hash,
           f.name AS source,
           COUNT(ui.id) AS interactions
    FROM articles a
    JOIN feeds f ON a.feed_id = f.id
    LEFT JOIN user_interactions ui ON a.id = ui.article_id
    GROUP BY a.id, f.name
    ORDER BY a.published_at DESC
""")
articles = cursor.fetchall()
conn.close()

# Score and rank
scored = []
for a in articles:
    f = freshness_score(a["published_at"])
    p = popularity_score(a["source_count"], a["interactions"])
    c = combined_score(a["published_at"], a["source_count"], a["interactions"])
    scored.append({**a, "freshness": f, "popularity": p, "score": c})

scored.sort(key=lambda x: x["score"], reverse=True)

print("📊 Article Rankings (freshness 60% + popularity 40%):")
print("=" * 95)
print(f"  {'#':>2} {'Title':<45} {'Source':<15} {'Fresh':>6} {'Pop':>6} {'Score':>6}")
print("-" * 95)
for i, a in enumerate(scored[:15]):
    print(f"  {i+1:>2} {a['title'][:44]:<45} {a['source'][:14]:<15} "
          f"{a['freshness']:>6.3f} {a['popularity']:>6.3f} {a['score']:>6.3f}")

## 🏆 Ranked Feed with Dedup

Now let's combine deduplication and ranking to produce a clean feed: **one article per story, ranked by combined score**.

In [ ]:
def build_ranked_feed(limit: int = 10) -> list:
    """
    Build a deduplicated, ranked feed:
    1. Fetch all articles
    2. Remove duplicates (keep canonical)
    3. Score and rank
    """
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Get articles with their duplicate group info
    cursor.execute("""
        SELECT a.id, a.title, a.published_at, a.source_count, a.summary,
               f.name AS source,
               COUNT(ui.id) AS interactions,
               dg.canonical_article_id
        FROM articles a
        JOIN feeds f ON a.feed_id = f.id
        LEFT JOIN user_interactions ui ON a.id = ui.article_id
        LEFT JOIN duplicate_members dm ON a.id = dm.article_id
        LEFT JOIN duplicate_groups dg ON dm.group_id = dg.id
        GROUP BY a.id, f.name, dg.canonical_article_id
    """)
    articles = cursor.fetchall()
    conn.close()

    # Filter: keep only canonical articles (or those not in any group)
    feed = []
    for a in articles:
        if a["canonical_article_id"] is None:  # not in a dup group
            feed.append(a)
        elif a["id"] == a["canonical_article_id"]:  # is the canonical
            feed.append(a)
        # else: skip (it's a duplicate, not canonical)

    # Score and sort
    for a in feed:
        a["score"] = combined_score(
            a["published_at"], a["source_count"], a["interactions"]
        )

    feed.sort(key=lambda x: x["score"], reverse=True)
    return feed[:limit]


feed = build_ranked_feed(limit=10)

print("📰 Deduplicated & Ranked Feed (Top 10):")
print("=" * 85)
for i, a in enumerate(feed):
    sources = f"({a['source_count']} source{'s' if a['source_count'] > 1 else ''})"
    print(f"\n  {i+1}. {a['title'][:60]}")
    print(f"     {a['source']} {sources} | score: {a['score']:.3f}")
    if a['summary']:
        print(f"     {a['summary'][:80]}...")

## ⚡ Caching Rankings in Redis

Recomputing the ranked feed for every request is expensive. We cache the global ranked feed in a **Redis sorted set** with a TTL, so most reads are instant.

In [ ]:
r = get_redis()
FEED_CACHE_KEY = "global_feed"
FEED_TTL_SECONDS = 300  # 5 minutes

def cache_ranked_feed(feed: list):
    """
    Store the ranked feed in a Redis sorted set.
    Score = ranking score (higher = better).
    Member = JSON blob with article data.
    """
    pipe = r.pipeline()
    pipe.delete(FEED_CACHE_KEY)

    for article in feed:
        member = json.dumps({
            "id": article["id"],
            "title": article["title"],
            "source": article["source"],
            "source_count": article["source_count"],
            "summary": (article["summary"] or "")[:200],
        })
        pipe.zadd(FEED_CACHE_KEY, {member: article["score"]})

    pipe.expire(FEED_CACHE_KEY, FEED_TTL_SECONDS)
    pipe.execute()


def get_cached_feed(offset: int = 0, limit: int = 5) -> list:
    """
    Read the ranked feed from Redis cache (highest score first).
    Returns empty list on cache miss.
    """
    results = r.zrevrange(FEED_CACHE_KEY, offset, offset + limit - 1, withscores=True)
    return [(json.loads(member), score) for member, score in results]


# Cache the feed we just built
cache_ranked_feed(feed)
print(f"✅ Cached {len(feed)} articles in Redis (TTL: {FEED_TTL_SECONDS}s)")

# Read it back — this is what a user request would do
cached = get_cached_feed(offset=0, limit=5)
print(f"\n📰 Reading from cache (page 1, 5 articles):")
print("=" * 70)
for i, (article, score) in enumerate(cached):
    sources = f"({article['source_count']} sources)" if article['source_count'] > 1 else ""
    print(f"  {i+1}. {article['title'][:55]}")
    print(f"     {article['source']} {sources} | score: {score:.3f}")

# Check TTL
ttl = r.ttl(FEED_CACHE_KEY)
print(f"\n⏰ Cache expires in {ttl} seconds")
print("💡 In production, refresh the cache every few minutes with a background job.")

## 🧹 Cleanup

In [ ]:
r = get_redis()
keys = r.keys("global_feed*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 📚 Summary

### Key Takeaways

1. **Content hashing** is exact dedup **after normalisation** — perfect for syndicated copies, useless for rewrites.
2. **Bag-of-words Jaccard** is a beginner-friendly fuzzy dedup: tokenise, drop stop-words, compute `|A ∩ B| / |A ∪ B|`.
3. **Shingling** captures local word order but is brittle under paraphrasing — complementary to word-set Jaccard.
4. **Greedy clustering + a "match the representative" guard** groups duplicates without over-merging via transitivity.
5. **Canonical selection** picks the best article per cluster (longest summary, earliest published).
6. **Exponential time decay** ensures fresh news ranks higher — freshness halves every `half_life` hours.
7. **Log-scale popularity** prevents viral articles from permanently dominating.
8. **Redis sorted sets** cache the ranked feed for sub-millisecond reads.
9. **LSH is a tuning problem, not a switch** — the (bands × rows) split moves an
   S-curve knee. Put the knee above your duplicates' similarity and you silently
   lose most of them; the sweep above shows 0.7% recall at 8×8 and 99.7% at 32×2
   on the same signatures.
10. **Detection is worthless until it's persisted** — write the clusters back,
    derive `source_count` from distinct feeds, and make the job idempotent so a
    scheduled re-run converges instead of accumulating.

### Bad → Best Progression

| Level | Technique | Catches | Misses |
|-------|-----------|---------|--------|
| Bad | `title_a == title_b` | Exact string matches | Everything else |
| Good | SHA-256 of normalised text | Republished / re-ingested articles | Any rewriting |
| Better | Bag-of-words Jaccard | Rewrites that share key terms | Topically-similar-but-different stories |
| Production | MinHash + LSH (+ SimHash) | Everything above, at O(n) scale | — |

### System Design Interview Tips

- **MinHash + LSH** is the go-to scaling answer — signatures make similarity estimation O(k) per pair; LSH narrows candidates to O(n).
- **Threshold tuning is empirical** — show a table of known positive/negative pairs to justify your cut-off.
- **Source count** is a strong popularity signal — a story reported by 10 outlets outranks one from a single blog.
- **Per-source trust score** is a natural next step — weight canonical selection and ranking by outlet reputation.

### Next Up

In **Notebook 3**, we'll build **Personalised Feed Generation** — using user interest profiles and reading history to tailor the feed to each user.
